# Asset Class Trend Following 策略回測 (equityV(固))

本報告針對 **2019-01-01 - 2025-12-31** 期間進行回測，並比較兩種 MDD 計算邏輯。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from backtest_v2 import clean_data, BacktesterV2, calculate_metrics

# 1. 資料讀取與清洗
prices, volumes, code_to_name = clean_data('樣本集-1.xlsx')

# 2. 設定參數與執行回測
sma, roc, sl, reb = 303, 14, 0.0999, 9
initial_capital = 30000000
fixed_base_capital = 150000000

bt = BacktesterV2(prices, volumes, code_to_name, initial_capital=initial_capital)
eq, trades, hold, trades2, daily = bt.run(sma, roc, sl, reb, 'peak', 10)

# 3. 篩選指定期間績效
mask = (eq['日期'] >= '2019-01-01') & (eq['日期'] <= '2025-12-31')
res = eq[mask].copy()

# 4. 計算固定基準 MDD
res['Peak_Equity'] = res['權益'].cummax()
res['Fixed_Base_Drawdown'] = (res['權益'] - res['Peak_Equity']) / fixed_base_capital
mdd_fixed = res['Fixed_Base_Drawdown'].min()

# 5. 繪製權益曲線
plt.figure(figsize=(12, 6))
plt.plot(res['日期'], res['權益'])
plt.title(f'Equity Curve (equityV(固))')
plt.grid(True)
plt.show()

# 6. 計算與顯示績效指標
cagr, mdd_std, calmar_std, total_ret = calculate_metrics(res)
calmar_fixed = cagr / abs(mdd_fixed) if mdd_fixed != 0 else 0

print(f"年化報酬率 (CAGR): {cagr:.2%}")
print(f"總報酬率: {total_ret:.2%}")
print(f"標準最大回撤 (Standard MaxDD): {mdd_std:.2%}")
print(f"固定基準最大回撤 (Fixed Base MaxDD): {mdd_fixed:.2%}")
print(f"Calmar Ratio (標準): {calmar_std:.2f}")
print(f"Calmar Ratio (固定基準): {calmar_fixed:.2f}")
